In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [3]:
NUC = "ACGT"
MOTIF = "TATA"
SEQ_LEN = 100
N_PAIRS = 2000 

In [4]:
def random_seq(n):
    return "".join(np.random.choice(list(NUC), n))

def make_positive():
    # Motifi rastgele bir konuma koy (sabit konum ezberlemeyi engeller)
    pos = np.random.randint(0, SEQ_LEN - len(MOTIF) + 1)
    left = random_seq(pos)
    right = random_seq(SEQ_LEN - pos - len(MOTIF))
    return left + MOTIF + right

def make_negative():
    # Motif içermeyen dizi üret
    while True:
        s = random_seq(SEQ_LEN)
        if MOTIF not in s:
            return s

seqs, labels = [], []
for _ in range(N_PAIRS):
    seqs.append(make_positive()); labels.append(1)
    seqs.append(make_negative()); labels.append(0)

In [5]:
CHAR2IDX = {c: i for i, c in enumerate(NUC)}

def one_hot(seq):
    arr = np.zeros((len(seq), len(NUC)), dtype="float32")
    arr[np.arange(len(seq)), [CHAR2IDX[c] for c in seq]] = 1.0
    return arr

X = np.stack([one_hot(s) for s in seqs])   # (N, 100, 4)
y = np.array(labels, dtype="float32")

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

In [7]:
model = keras.Sequential([
    keras.Input(shape=(SEQ_LEN, len(NUC))),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(name="auc"), "accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)        │ (None, 64)                  │           9,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 16)                  │           1,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 10,529 (41.13 KB)

 Trainable params: 10,529 (41.13 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=64,
    callbacks=[early_stop],
    verbose=2,
)

Epoch 1/40
40/40 - 16s - 389ms/step - accuracy: 0.5152 - auc: 0.5183 - loss: 0.6925 - val_accuracy: 0.5250 - val_auc: 0.5705 - val_loss: 0.6884
Epoch 2/40
40/40 - 8s - 199ms/step - accuracy: 0.5582 - auc: 0.5749 - loss: 0.6855 - val_accuracy: 0.5484 - val_auc: 0.6243 - val_loss: 0.6769
Epoch 3/40
40/40 - 5s - 128ms/step - accuracy: 0.5813 - auc: 0.6109 - loss: 0.6751 - val_accuracy: 0.5844 - val_auc: 0.6400 - val_loss: 0.6718
Epoch 4/40
40/40 - 5s - 128ms/step - accuracy: 0.5965 - auc: 0.6332 - loss: 0.6658 - val_accuracy: 0.6094 - val_auc: 0.6536 - val_loss: 0.6603
Epoch 5/40
40/40 - 4s - 92ms/step - accuracy: 0.6043 - auc: 0.6406 - loss: 0.6638 - val_accuracy: 0.6094 - val_auc: 0.6581 - val_loss: 0.6580
Epoch 6/40
40/40 - 4s - 91ms/step - accuracy: 0.6078 - auc: 0.6442 - loss: 0.6624 - val_accuracy: 0.6203 - val_auc: 0.6643 - val_loss: 0.6573
Epoch 7/40
40/40 - 5s - 130ms/step - accuracy: 0.5992 - auc: 0.6409 - loss: 0.6636 - val_accuracy: 0.6141 - val_auc: 0.6708 - val_loss: 0.6592


In [9]:
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(f"Test ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(classification_report(y_test, y_pred,
                            target_names=["non-promoter", "promoter"]))

Test ROC-AUC: 0.651
              precision    recall  f1-score   support

non-promoter       0.66      0.43      0.52       400
    promoter       0.58      0.78      0.66       400

    accuracy                           0.60       800
   macro avg       0.62      0.60      0.59       800
weighted avg       0.62      0.60      0.59       800



In [10]:
def predict_seq(seq):
    p = model.predict(one_hot(seq)[None, ...], verbose=0)[0, 0]
    return int(p > 0.5), float(p)

examples = {
    "TATA ortada":      "A" * 40 + "TATA" + "C" * 56,
    "TATA yok (ACGT)":  "ACGT" * 25,
}
for name, s in examples.items():
    label, p = predict_seq(s)
    print(f"{name:20s} -> sınıf={label}, olasılık={p:.3f}")

TATA ortada          -> sınıf=1, olasılık=0.532
TATA yok (ACGT)      -> sınıf=0, olasılık=0.484


In [11]:
model.save("../model/bio_rnn.keras")